In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

%cd /content/drive/MyDrive/LLM/Bioreasoner/testing_place_v2/src


Mounted at /content/drive
/content/drive/MyDrive/LLM/Bioreasoner/testing_place_v2/src


In [2]:
%cd /content/drive/MyDrive/LLM/Bioreasoner/testing_place_v2/src

/content/drive/MyDrive/LLM/Bioreasoner/testing_place_v2/src


In [3]:
# If running on Colab, install minimal deps
# (Comment out torch install if Colab runtime already has a good CUDA build.)
# %pip install -q --upgrade transformers accelerate

import os, sys, platform, torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.8.0+cu126
CUDA available: True


In [4]:
!pip install llamafactory

In [5]:
#!pip -q install --upgrade "transformers>=4.43" accelerate safetensors sentencepiece einops
import torch, platform, sys, os, textwrap
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


Torch: 2.8.0+cu126 | CUDA: True | Device: NVIDIA A100-SXM4-80GB


In [6]:
# Versions + sanity checks (run after installing deps)

import sys, os, platform, tempfile, warnings
import torch
from packaging import version

# Core libs
import transformers, accelerate, safetensors, sentencepiece, einops

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("Torch:", torch.__version__, "| CUDA avail:", torch.cuda.is_available(), "| CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU CC:", f"{props.major}.{props.minor}", "| Total VRAM (GB):", round(props.total_memory/1024**3, 2))

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("safetensors:", safetensors.__version__)
print("sentencepiece:", sentencepiece.__version__)
print("einops:", einops.__version__)

# Assert min versions you requested
assert version.parse(transformers.__version__) >= version.parse("4.43"), "Transformers >= 4.43 required"


Python: 3.12.12 | Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Torch: 2.8.0+cu126 | CUDA avail: True | CUDA: 12.6
GPU: NVIDIA A100-SXM4-80GB
GPU CC: 8.0 | Total VRAM (GB): 79.32
Transformers: 4.52.4
Accelerate: 1.7.0
safetensors: 0.6.2
sentencepiece: 0.2.1
einops: 0.8.1


In [7]:
BASE_MODEL_NAME        = "Qwen/Qwen2.5-0.5B-Instruct"     # LLM backbone
PROTEIN_CONFIG = "/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/esm2_t12_35M_UR50D"
STRUCTURE_CONFIG = "/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/foldseek_t12_35M"
PROTREK_CKPT_PATH    = "/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/ProTrek_35M.pt"


DTYPE = "bf16" if torch.cuda.is_available() else "fp16"
print("Using dtype:", DTYPE)


Using dtype: bf16


In [8]:
#import protein_llm  # this file must define class `PLLM`
import protein_llm.models.modeling_pllm

[WARN] flash_attn is not installed, flash_attn will not work


In [9]:
import os, json, math
from typing import Optional, List
import torch
from transformers import AutoTokenizer

# Import your updated PLLM module (must be in the same folder)
import protein_llm  # this file must define class `PLLM`
import protein_llm.models.modeling_pllm as PLLM
from protein_llm.models.configuration_pllm import PLLMConfig

# ===== USER-EDITABLE PATHS =====
#BASE_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # or "Qwen/Qwen2.5-0.5B"
PROT_SLOT = 1
STRU_SLOT = 3

# BF16 is optimal for A100
DTYPE_STR = "bf16"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [10]:
from transformers import AutoTokenizer

BASE_LLM = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE_LLM, use_fast=True)

# Chain/seq/struct markers (LLM tokens only; encoders see raw strings)
special_tokens = ["<chain_bos>", "<chain_eos>", "<seq_bos>", "<seq_eos>", "<struct_bos>", "<struct_eos>"]
tokenizer.add_tokens(special_tokens)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

chain_bos_id   = tokenizer.convert_tokens_to_ids("<chain_bos>")
chain_eos_id   = tokenizer.convert_tokens_to_ids("<chain_eos>")
seq_bos_id     = tokenizer.convert_tokens_to_ids("<seq_bos>")
seq_eos_id     = tokenizer.convert_tokens_to_ids("<seq_eos>")
struct_bos_id  = tokenizer.convert_tokens_to_ids("<struct_bos>")
struct_eos_id  = tokenizer.convert_tokens_to_ids("<struct_eos>")

# Encoders: light ESM configs for a quick test; set PROTREK_CKPT to your local ckpt to load real weights
BASE_MODEL_NAME        = "Qwen/Qwen2.5-0.5B-Instruct"     # LLM backbone
PROTEIN_CONFIG = "/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/esm2_t12_35M_UR50D"
STRUCTURE_CONFIG = "/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/foldseek_t12_35M"
PROTREK_CKPT_PATH    = "/content/drive/MyDrive/LLM/Bioreasoner/protrek/weights/ProTrek_35M/ProTrek_35M.pt"

config = PLLMConfig(
    base_model_name_or_path=BASE_MODEL_NAME,
    protein_config=PROTEIN_CONFIG,
    structure_config=STRUCTURE_CONFIG,
    protrek_ckpt=PROTREK_CKPT_PATH,
    prot_slot=1,
    stru_slot=3,
    train_encoders=True,   # set False to freeze encoders
    proj_hid=1024,
    dropout=0.10,
    # NEW chain-aware IDs
    chain_bos_id=chain_bos_id,
    chain_eos_id=chain_eos_id,
    seq_bos_id=seq_bos_id,
    seq_eos_id=seq_eos_id,
    struct_bos_id=struct_bos_id,
    struct_eos_id=struct_eos_id,
)

model = PLLM.PLLM(config).to(device)
model.llm.resize_token_embeddings(len(tokenizer))  # IMPORTANT after adding tokens

if PROTREK_CKPT_PATH:
    model.load_protrek_weights()

print("[ok] model on", device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[PLLM] Loaded generation_config from base model: Qwen/Qwen2.5-0.5B-Instruct
[ProteinEncoder] loaded from slot 1 | missing=[] unexpected=[]
[StructureEncoder] loaded from slot 3 | missing=[] unexpected=[]
[ok] model on cuda


In [11]:
test_item = {
    "pdb_id": "3M2K",
    "chains": {
        "A": {
            "aa_seq": "DDFAKLEEQFDAKLGIFALDTGTNRTVAYRPDERFAFASTIKALTVGVLLQQKSIEDLNQRITYTRDDLVNYNPITEKHVDTGMTLKELADASLRYSDNAAQNLILKQIGGPESLKKELRKIGDEVTNPEGETQDTSTARALVTSLRAFALEDKLPSEKRELLIDWMKRNTTGDALIRAGVPDGWEVADKTGAASYGTRNDIAIIWPPKGDPVVLAVLSSRDKKDAKYDDKLIAEATKVVMKALN",
            "threeDi_seq": "DVVVVVCVVFVKDKFKWKAFPVVGDIDGDPFFDKDFQALQCLLVLVLVLQLPADVVQQQPFDADDVVLDDDFAPPNVVPNPPTGGNLRLNLCCFQVVGPSSSVVSQVVVPHQVSSVVVVVVVVDPFQADPVDPHRIGGFVVVVVSVCCRQPNDSHDPVSSVSSQVSQQNRNPDPLALVLQADPPKDKRKGWRAFFQAKTKMWIWIHDVPDTIMTIIMIMGHDHGNDHYDRCSRNVVNNCVVVVPD"
        },
        "B": {
            "aa_seq": "KDDFAKLEEQFDAKLGIFALDTGTNRTVAYRPDERFAFASTIKALTVGVLLQQKSIEDLNQRITYTRDDLVNYNPITEKHVDTGMTLKELADASLRYSDNAAQNLILKQIGGPESLKKELRKIGDEVTNPERFVNPGETQDTSTARALVTSLRAFALEDKLPSEKRELLIDWMKRNTTGDALIRAGVPDGWEVADKTGAASYGTRNDIAIIWPPKGDPVVLAVLSSRDKKDAKYDDKLIAEATKVVMKALN",
            "threeDi_seq": "DVQLVVVCVVFVWDKAKWKAFVVPRDIDDDNQPDKDLQFLVCLLLLVLVLLLPADVVRLQAFDAQDVPQQDPFAPPVVVDNVRGGRLLVLNLCCFQVVGLSSSLVSCVVQVHQVSSVVVLVVQPQPFAAEDDNCDPPDRYNMGGNVSVVVSVCCSQVNVPHDPVSSVSSQVSQQNRPPDCLAVVLLDDPPKGKRKDWRAFWQQKTWMWIWIADPDGDIMTMIMIMGHDHGNDHYDRNSRSVVSNVVVVVVD"
        }
    }
}

def build_chain_prompt(chains_dict):
    def one(letter, aa, st):
        return (f"<chain_bos>Chain {letter}："
                f"<seq_bos> {aa} <seq_eos> "
                f"<struct_bos> {st} <struct_eos>"
                f"<chain_eos>")
    return " ".join([one(L, chains_dict[L]["aa_seq"], chains_dict[L]["threeDi_seq"])
                     for L in sorted(chains_dict.keys())])

user_prompt = (
    "You are a professional protein biologist. Based only on the provided inputs, "
    "produce a natural, concise, and biologically accurate description of the protein. "
    "First reason step by step inside a <thinking> block using sequence-derived evidence and structural cues, "
    "then provide the final 2–4 sentence description inside an <answer> block.\n\n"
    "Inputs:\n" + build_chain_prompt(test_item["chains"])
)

ref_answer = (
    "<thinking>Token-level encoders suggest an enzyme-like fold with catalytic residues in conserved motifs.</thinking>\n\n"
    "<answer>Class A beta-lactamase-like enzyme; flexible Ω-loop may accommodate bulky substrates.</answer>"
)

def make_sft_batch(tok, prompt, response):
    p = tok.encode(prompt, add_special_tokens=False)
    r = tok.encode(response, add_special_tokens=False)
    input_ids = torch.tensor([p + r + [tok.eos_token_id]], device=device)
    labels    = torch.tensor([[-100]*len(p) + r + [-100]], device=device)
    attn      = torch.ones_like(input_ids)
    return input_ids, attn, labels

input_ids, attention_mask, labels = make_sft_batch(tokenizer, user_prompt, ref_answer)

# per-sample chain lists (order must match the prompt)
aa_seq  = [[ test_item["chains"]["A"]["aa_seq"],       test_item["chains"]["B"]["aa_seq"] ]]
stru_str= [[ test_item["chains"]["A"]["threeDi_seq"],  test_item["chains"]["B"]["threeDi_seq"] ]]

input_ids.shape, attention_mask.shape, labels.shape


(torch.Size([1, 675]), torch.Size([1, 675]), torch.Size([1, 675]))

In [12]:
model.train()
with torch.cuda.amp.autocast(enabled=torch.cuda.is_available(),
                             dtype=torch.bfloat16 if torch.cuda.is_available() else None):
    out = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        aa_seq=aa_seq,
        stru_str=stru_str,
    )
print("forward loss:", float(out.loss))


/tmp/ipython-input-1451427748.py:2: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available(),
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


forward loss: 4.234283924102783


/tmp/ipython-input-1451427748.py:11: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  print("forward loss:", float(out.loss))


In [13]:
# --- No AMP, No Accelerate: pure PyTorch smoke test (forward → backward → step → export_qkv) ---

import torch
from torch.optim import AdamW

# (Optional) turn off gradient checkpointing to avoid "inputs have requires_grad=False" warnings in this smoke test
getattr(model.llm, "gradient_checkpointing_disable", lambda **kwargs: None)()

model.train()
opt = AdamW((p for p in model.parameters() if p.requires_grad), lr=5e-5)
opt.zero_grad(set_to_none=True)

# Forward (fp32 unless your weights are bf16/fp16 already)
out = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    aa_seq=aa_seq,
    stru_str=stru_str,
)
loss = out.loss
print("forward loss (no amp):", loss.detach().float().item())

# Backward + update
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
opt.step()
opt.zero_grad(set_to_none=True)
print("did one optimizer step")

# ---- export_qkv test (no accelerator wrapping) ----
# Run a capture pass with extract_qkv=True; choose any valid layer indices
_ = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
    aa_seq=aa_seq,
    stru_str=stru_str,
    extract_qkv=True,
    layer_idx=[0, 3],   # change as you like
)

pack = model.export_qkv(split_heads=True)
layers = list(pack.get("layers", {}).keys())
print("captured layers:", layers)

for li, entry in pack.get("layers", {}).items():
    q, k, v, meta = entry["q"], entry["k"], entry["v"], entry["meta"]
    print(f"L{li}: q{tuple(q.shape)} k{tuple(k.shape)} v{tuple(v.shape)} | "
          f"heads={meta.get('num_heads')} kv_heads={meta.get('num_key_value_heads')}")

print("mask shape:", None if pack.get("m") is None else tuple(pack["m"].shape))


forward loss (no amp): 4.225318908691406
did one optimizer step
captured layers: [0, 3]
L0: q(1, 627, 14, 64) k(1, 627, 2, 64) v(1, 627, 2, 64) | heads=14 kv_heads=2
L3: q(1, 627, 14, 64) k(1, 627, 2, 64) v(1, 627, 2, 64) | heads=14 kv_heads=2
mask shape: (1, 627)


In [14]:
layers_to_probe = [0, 3]  # pick any valid layer indices for Qwen2.5-0.5B
with torch.no_grad():
    _ = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        aa_seq=aa_seq,
        stru_str=stru_str,
        extract_qkv=True,
        layer_idx=layers_to_probe,
    )

pack = model.export_qkv(split_heads=True)
print("captured layers:", list(pack["layers"].keys()))
for li, entry in pack["layers"].items():
    q, k, v, meta = entry["q"], entry["k"], entry["v"], entry["meta"]
    print(f"L{li} -> q{tuple(q.shape)} k{tuple(k.shape)} v{tuple(v.shape)} | "
          f"heads={meta['num_heads']} kv_heads={meta['num_key_value_heads']}")
print("mask shape:", None if pack["m"] is None else tuple(pack["m"].shape))


captured layers: [0, 3]
L0 -> q(1, 627, 14, 64) k(1, 627, 2, 64) v(1, 627, 2, 64) | heads=14 kv_heads=2
L3 -> q(1, 627, 14, 64) k(1, 627, 2, 64) v(1, 627, 2, 64) | heads=14 kv_heads=2
mask shape: (1, 627)


In [15]:
from torch.optim import AdamW
opt = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)

model.train()
opt.zero_grad(set_to_none=True)
out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels,
            aa_seq=aa_seq, stru_str=stru_str)
loss = out.loss
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
opt.step(); opt.zero_grad(set_to_none=True)
print("one-step SFT loss:", float(loss))


one-step SFT loss: 1.711367130279541
